# Models

> Data models for the graph management interface

In [ ]:
#| default_exp models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

## SegmentSample

A lightweight snapshot of a single segment for display in detail views.

In [ ]:
#| export
@dataclass
class SegmentSample:
    """Lightweight segment snapshot for detail view display."""
    index: int          # Segment position in the document
    text: str           # Segment text content
    start_time: float   # Start time in seconds
    end_time: float     # End time in seconds

In [ ]:
s = SegmentSample(index=0, text="Sun Tzu said,", start_time=0.0, end_time=2.1)
assert s.index == 0
assert s.end_time - s.start_time == 2.1
print(f"[{s.index}] \"{s.text}\" ({s.start_time}s - {s.end_time}s)")

[0] "Sun Tzu said," (0.0s - 2.1s)


## DocumentSummary

Summary row for the document list table. One per document.

In [ ]:
#| export
@dataclass
class DocumentSummary:
    """Summary of a single document for list display."""
    document_id: str        # Document node UUID
    title: str              # Document title from properties
    media_type: str         # e.g. "audio"
    segment_count: int      # Number of Segment nodes
    total_duration: float   # Sum of segment durations in seconds
    created_at: float       # Unix timestamp when created

In [ ]:
doc = DocumentSummary(
    document_id="abc-123", title="1. Laying Plans",
    media_type="audio", segment_count=247,
    total_duration=2537.0, created_at=1740000000.0
)
assert doc.segment_count == 247
print(f"{doc.title}: {doc.segment_count} segments, {doc.total_duration:.0f}s")

1. Laying Plans: 247 segments, 2537s


## DocumentDetail

Full document information for the detail view, including integrity checks and sample segments.

In [ ]:
#| export
@dataclass
class DocumentDetail:
    """Full document information for the detail dashboard."""
    # Document info
    document_id: str        # Document node UUID
    title: str              # Document title
    media_type: str         # e.g. "audio"
    created_at: float       # Unix timestamp
    updated_at: float       # Unix timestamp

    # Segment stats
    segment_count: int              # Total number of segments
    total_duration: float           # Sum of segment durations in seconds
    avg_segment_duration: float     # Average segment duration in seconds

    # Integrity checks
    has_starts_with: bool           # Document has a STARTS_WITH edge
    next_chain_complete: bool       # All NEXT edges form a complete chain
    next_count: int                 # Number of NEXT edges found
    part_of_complete: bool          # All segments have PART_OF edges
    part_of_count: int              # Number of PART_OF edges found
    all_have_timing: bool           # All segments have start_time/end_time
    segments_missing_timing: int    # Count of segments without timing
    all_have_sources: bool          # All segments have source references
    segments_missing_sources: int   # Count of segments without sources
    all_checks_passed: bool         # True if all integrity checks pass

    # Source info
    source_plugins: List[str] = field(default_factory=list)  # Unique plugin names from sources

    # Sample segments
    first_segments: List[SegmentSample] = field(default_factory=list)  # First N segments
    last_segments: List[SegmentSample] = field(default_factory=list)   # Last N segments

In [ ]:
detail = DocumentDetail(
    document_id="abc-123", title="1. Laying Plans",
    media_type="audio", created_at=1740000000.0, updated_at=1740000000.0,
    segment_count=247, total_duration=2537.0, avg_segment_duration=10.3,
    has_starts_with=True, next_chain_complete=True, next_count=246,
    part_of_complete=True, part_of_count=247,
    all_have_timing=True, segments_missing_timing=0,
    all_have_sources=True, segments_missing_sources=0,
    all_checks_passed=True,
    source_plugins=["cjm-transcription-plugin-whisper"],
    first_segments=[SegmentSample(0, "Sun Tzu said,", 0.0, 2.1)],
    last_segments=[SegmentSample(246, "Hence under these five...", 2528.0, 2537.0)],
)
assert detail.all_checks_passed
assert detail.next_count == detail.segment_count - 1
print(f"{detail.title}: {detail.segment_count} segments, all checks passed: {detail.all_checks_passed}")

1. Laying Plans: 247 segments, all checks passed: True


## ExportBundle

Metadata wrapper for exported graph data. The `graph` field contains the raw `{"nodes": [...], "edges": [...]}` structure.

In [ ]:
#| export
@dataclass
class ExportBundle:
    """Metadata wrapper for exported graph data."""
    format: str              # Always "cjm-context-graph"
    version: str             # Semantic version, e.g. "1.0.0"
    exported_at: str         # ISO 8601 datetime string
    source_plugin: str       # Plugin that produced the data
    document_count: int      # Number of Document nodes in the export
    graph: Dict[str, Any]    # {"nodes": [...], "edges": [...]}

In [ ]:
bundle = ExportBundle(
    format="cjm-context-graph", version="1.0.0",
    exported_at="2026-02-19T12:00:00Z",
    source_plugin="cjm-graph-plugin-sqlite",
    document_count=1,
    graph={"nodes": [], "edges": []}
)
assert bundle.format == "cjm-context-graph"
print(f"Export: {bundle.format} v{bundle.version}, {bundle.document_count} doc(s)")

Export: cjm-context-graph v1.0.0, 1 doc(s)


## ImportResult

Result of a graph import operation.

In [ ]:
#| export
@dataclass
class ImportResult:
    """Result of a graph import operation."""
    success: bool           # Whether the import succeeded
    nodes_created: int      # Number of nodes created
    edges_created: int      # Number of edges created
    nodes_skipped: int      # Number of nodes skipped (already exist)
    edges_skipped: int      # Number of edges skipped (already exist)
    errors: List[str] = field(default_factory=list)  # Error messages if any

In [ ]:
result = ImportResult(
    success=True, nodes_created=248, edges_created=494,
    nodes_skipped=0, edges_skipped=0
)
assert result.success
assert len(result.errors) == 0
print(f"Import: {result.nodes_created} nodes, {result.edges_created} edges")

Import: 248 nodes, 494 edges


## ManagementUrls

URL bundle for all management routes. Constructed during router assembly and passed to components for link generation.

In [ ]:
#| export
@dataclass
class ManagementUrls:
    """URL bundle for management route endpoints."""
    list_documents: str     # GET: document list page
    document_detail: str    # GET: + /{doc_id}
    delete_document: str    # DELETE: + /{doc_id}
    delete_selected: str    # POST: bulk delete
    export_document: str    # GET: + /{doc_id}
    export_all: str         # GET: full database export
    import_graph: str       # POST: file upload import

In [ ]:
urls = ManagementUrls(
    list_documents="/manage/documents",
    document_detail="/manage/documents/detail",
    delete_document="/manage/documents/delete",
    delete_selected="/manage/documents/delete-selected",
    export_document="/manage/export/document",
    export_all="/manage/export/all",
    import_graph="/manage/import",
)
assert urls.list_documents.startswith("/manage")
print(f"List URL: {urls.list_documents}")
print(f"Detail URL: {urls.document_detail}/{{doc_id}}")

List URL: /manage/documents
Detail URL: /manage/documents/detail/{doc_id}


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()